In [ ]:
#pip install openpyxl
#pip install pandas

import pandas as pd

#Parametro caminho arquivo
arquivo_excel = 'dados/extrato.xlsx'  # extrato exportado da corretora (nao versionado)

#Busca arquivo
df = pd.read_excel(arquivo_excel)
df.head()


In [ ]:
# Célula 2: Inserção de colunas e finalização da estrutura

# 1. Lista das colunas que você deseja inserir
novas_colunas = ['Categoria', 'Rendimento', 'Ação', 'Fator', 'Tipo Movimentação']

# 2. Adiciona as colunas ao DataFrame 'df' que já está na memória
# Usamos um loop para criar todas como vazias de uma vez
for coluna in novas_colunas:
    if coluna not in df.columns:
        df[coluna] = ""

# 3. Reorganização (Opcional): Caso você queira garantir que elas fiquem no final
colunas_finais = [c for c in df.columns if c not in novas_colunas] + novas_colunas
df = df[colunas_finais]

# 4. Salva o arquivo final atualizado
df.to_excel(caminho_saida, index=False)

print("Estrutura atualizada com sucesso!")
print(f"Colunas atuais: {list(df.columns)}")
df.head() # Mostra o resultado na tela do Notebook

In [ ]:
import pandas as pd
import numpy as np

# 1. Lê o arquivo original pulando o cabeçalho administrativo
df = pd.read_excel(caminho_entrada, skiprows=13)

# 2. Limpeza do rodapé (baseada na coluna B que agora é o índice 1)
nulos_coluna_b = df[df.iloc[:, 1].isna()].index
if not nulos_coluna_b.empty:
    df = df.iloc[:nulos_coluna_b[0]]

# 3. Adiciona as novas colunas (sem deletar as originais)
novas_colunas = ['Categoria', 'Rendimento', 'Ação', 'Fator', 'Tipo Movimentação']
for coluna in novas_colunas:
    if coluna not in df.columns:
        df[coluna] = ""

# 4. Atualiza o FATOR (Equivalente ao SE do Excel)
# No seu arquivo, a coluna de Valor é a 6ª (índice 5)
coluna_valor_nome = df.columns[5] 

# Converte para numérico para garantir que o cálculo > 0 funcione
valor_numerico = pd.to_numeric(df[coluna_valor_nome], errors='coerce')
df['Fator'] = np.where(valor_numerico > 0, 'POSITIVO', 'NEGATIVO')

# 5. Salva o resultado mantendo tudo
df.to_excel(caminho_saida, index=False)

print("Processo concluído! Todas as colunas originais foram mantidas.")
print(f"Colunas atuais: {list(df.columns)}")
df.head()

In [ ]:
# Célula: Atualização do Tipo de Movimentação

# 1. Identifica a coluna de valor (no seu arquivo original mantido, é o índice 5)
coluna_valor_nome = df.columns[5]

# 2. Converte para numérico por segurança (evita erro com R$ ou textos)
valor_numerico = pd.to_numeric(df[coluna_valor_nome], errors='coerce')

# 3. Aplica a lógica: SE valor > 0 "RECEITA", senão "DESPESA"
df['Tipo Movimentação'] = np.where(valor_numerico > 0, 'RECEITA', 'DESPESA')

# 4. Salva o progresso no arquivo limpo
df.to_excel(caminho_saida, index=False)

print("Coluna 'Tipo Movimentação' preenchida com sucesso!")
# Exibe as colunas relevantes para conferência
print(df[[coluna_valor_nome, 'Fator', 'Tipo Movimentação']].head(10))

In [ ]:
# Célula: Automação Seguindo o Padrão do Usuário (Dicionário de Ativos e Categorias)
import re
import pandas as pd
import numpy as np

# 1. Função de categorização ajustada para os índices corretos
def categorizar_completo(linha):
    # Coluna Lançamento (Descrição) está no índice 3
    desc = str(linha[df.columns[3]]).upper() 
    # Coluna Valor está no índice 5
    valor = pd.to_numeric(linha[df.columns[5]], errors='coerce')
    
    categoria = "Outros"
    rendimento = ""
    acao = ""
    
    # Extração do código do ativo (ex: PETR4, KNRI11)
    ativo_match = re.search(r'[A-Z]{4}\d{1,2}', desc)
    if ativo_match:
        acao = ativo_match.group()
    
    # Regras de Categoria e Rendimento
    if any(x in desc for x in ["RENDIMENTOS", "DIVIDENDOS", "JUROS S/ CAPITAL"]):
        categoria = "Rendimento"
        rendimento = "S"
    elif "COMPRA" in desc:
        categoria = "Compra"
    elif "VENDA" in desc:
        categoria = "Venda"
    elif any(x in desc for x in ["TAXA", "DEBITO CBLC", "CUSTO"]):
        categoria = "Taxas"
    elif "TRANSFERENCIA" in desc:
        categoria = "Movimentação"
        
    return pd.Series([categoria, rendimento, acao])

# 2. Aplica a função (Categoria, Rendimento, Açao)
df[['Categoria', 'Rendimento', 'Açao']] = df.apply(categorizar_completo, axis=1)

# 3. Ajuste para CDB (Tratando a coluna como string para evitar o erro de datetime)
# Usamos o índice 3 (Lançamento)
mask_cdb = df[df.columns[3]].astype(str).str.contains('CDB', case=False, na=False)
df.loc[mask_cdb, 'Açao'] = df.loc[mask_cdb, df.columns[3]].apply(
    lambda x: "CDB " + str(x).split('CDB')[-1].strip() if 'CDB' in str(x) else x
)

# 4. Salva o resultado
df.to_excel(caminho_saida, index=False)

print("Classificação concluída com sucesso!")
# Exibe para conferência
df[[df.columns[3], 'Categoria', 'Rendimento', 'Açao']].head(10)

In [ ]:
import pandas as pd
import numpy as np
import re
import os

# CONFIGURAÇÕES DE CAMINHO
caminho_entrada = 'dados/extrato.xlsx'
caminho_saida = 'dados/extrato-limpo.xlsx'

def processar_extrato():
    try:
        if not os.path.exists(caminho_entrada):
            print(f"Erro: Arquivo não encontrado em {caminho_entrada}")
            return

        # 1. LEITURA
        df = pd.read_excel(caminho_entrada, skiprows=13)

        # 2. LIMPEZA DE RODAPÉ
        nulos_coluna_b = df[df.iloc[:, 1].isna()].index
        if not nulos_coluna_b.empty:
            df = df.iloc[:nulos_coluna_b[0]].copy()

        # 3. DICIONÁRIO DE CATEGORIAS (Com prioridade para IRRF)
        regras_categorias = {
            'IRRF': 'IR',
            'TAXA DE INTERMEDIAÇÃO DOADOR': 'Taxa de intermediação',
            'TAXA DE REMUNERAÇÃO': 'Taxa de Remuneração',
            'ESTORNO PREVIA CHAMADA DE MARGEM': 'Margem',
            'ESTORNO DE OPERAÇÕES': 'Estorno',
            'DÉBITO SUBSCRIÇÃO': 'Débito Subscrição',
            'SUBSCRIÇÃO': 'Subscrição',
            'RESGATE TREND INVESTBACK': 'Investback',
            'INVESTBACK': 'Investback',
            'INTEGRALIZAÇÃO DE COTAS': 'Operação Bolsa',
            'OPERAÇÕES EM BOLSA': 'Operação Bolsa',
            'FRAÇÕES DE AÇÕES': 'Dividendo',
            'JUROS S/ CAPITAL': 'Dividendo',
            'PGTO JUROS': 'Dividendo',
            'RENDIMENTO': 'Dividendo',
            'DIVIDENDOS': 'Dividendo',
            'REEMB. EVENTO': 'Reembolso',
            'REEMBOLSO': 'Reembolso',
            'TRANSFERÊNCIA ENVIADA PARA A CONTA DIGITAL': 'Pagamento CC',
            'TRANSFERÊNCIA RECEBIDA DA CONTA DIGITAL': 'Transferência',
            'RECEBIMENTO DE TED': 'Transferência',
            'RETIRADA EM C/C': 'Retirada',
            'MARGEM': 'Margem',
            'COMPRA': 'Compra',
            'IR': 'IR',
            'EMOLUMENTOS': 'Emolumentos',
            'IOF': 'IOF',
            'MULTA': 'Multa',
            'PAGAMENTO': 'Pagamento',
            'RESGATE': 'Resgate',
            'APLICAÇÃO': 'Aplicação',
            'VENCIMENTO': 'Vencimento'
        }

        def extrair_acao(desc):
            desc = str(desc).upper().replace('  ', ' ') # Remove espaços duplos
            
            # 1. Padrão BTC (Ex: BTC B3SA)
            btc_match = re.search(r'BTC\s+[A-Z0-9]{4,}', desc)
            if btc_match: return btc_match.group()

            # 2. Fundo Fechado Balcão (Ex: Valora CRI, XP Habitat FII)
            if 'BALCÃO' in desc:
                # Pega o que vem após "BALCÃO", extraindo as duas ou três primeiras palavras relevantes
                texto_pos_balcao = desc.split('BALCÃO')[-1].strip()
                # Remove códigos numéricos longos do final se existirem
                texto_limpo = re.sub(r'\d{5,}.*', '', texto_pos_balcao).strip()
                # Para "Valora CRI CDI Renda", pegamos apenas "Valora CRI"
                if 'VALORA CRI' in texto_limpo: return 'Valora CRI'
                # Para outros FIIs de balcão
                fii_match = re.search(r'(.*?FII)', texto_limpo)
                if fii_match: return fii_match.group(1).strip()
                return texto_limpo.split(' - ')[0].strip()

            # 3. Tesouro Direto
            if 'TESOURO DIRETO' in desc: return 'TESOURO DIRETO'
            
            # 4. Fundos
            if 'FUNDOS' in desc:
                match = re.search(r'FUNDOS\s+.*?(?=FIC FI|$)(\s*FIC FI)?', desc)
                if match: return match.group().strip()

            # 5. LCA/LCI (Emissor após o pipe |)
            if 'LCA' in desc or 'LCI' in desc:
                match = re.search(r'\|\s*(LC[A|I]\s+.*?)(?=-|\d{2}/|$)', desc)
                if match: return match.group(1).strip()

            # 6. Ativos de Bolsa e Códigos BR
            ticker_match = re.search(r'\b[A-Z]{4}\d{1,2}\b|\bBR[A-Z0-9]{10,}\b', desc)
            if ticker_match: return ticker_match.group()

            # 7. CDBs
            if 'CDB' in desc:
                return "CDB " + desc.split('CDB')[-1].strip()

            return ""

        def aplicar_logica_completa(linha):
            desc = str(linha.iloc[3]).upper()
            valor_num = pd.to_numeric(linha.iloc[5], errors='coerce')
            
            # --- CATEGORIA ---
            categoria = "Outros"
            for termo, label in regras_categorias.items():
                if termo in desc:
                    categoria = label
                    break
            
            # --- RENDIMENTO (G) ---
            cats_rendimento = ['Dividendo', 'Investback', 'Reembolso', 'Taxa de Remuneração', 'Liquidação']
            rendimento = "S" if (valor_num > 0 and categoria in cats_rendimento) else ""
            
            # --- AÇÃO / ATIVO (H) ---
            acao = extrair_acao(desc)

            # --- FATOR E TIPO ---
            fator = "POSITIVO" if valor_num > 0 else "NEGATIVO"
            tipo_mov = "RECEITA" if valor_num > 0 else "DESPESA"

            return pd.Series([categoria, rendimento, acao, fator, tipo_mov])

        # 4. APLICAÇÃO
        df[['Categoria', 'Rendimento', 'Açao', 'fator', 'Tipo Movimentação']] = df.apply(aplicar_logica_completa, axis=1)

        # 5. REMOÇÃO DAS COLUNAS A E E
        df = df.drop(columns=[df.columns[0], df.columns[4]])

        # 6. ORGANIZAÇÃO (Açao na Coluna H)
        ordem_final = [
            df.columns[0], df.columns[1], df.columns[2], df.columns[3], df.columns[4],
            'Categoria', 'Rendimento', 'Açao', 'fator', 'Tipo Movimentação'
        ]
        df = df[ordem_final]

        # 7. SALVAR
        df.to_excel(caminho_saida, index=False)
        print("Sucesso!")

    except Exception as e:
        print(f"Erro: {e}")

if __name__ == "__main__":
    processar_extrato()